### First we will load the dataset


In [12]:
import pandas as pd
import numpy as np


In [13]:
df = pd.read_csv("drug200.csv")

In [14]:
df.head()

,Age,Sex,BP,Cholesterol,Na_to_K,Drug
0,23,F,HIGH,HIGH,25.355,drugY
1,47,M,LOW,HIGH,13.093,drugC
2,47,M,LOW,HIGH,10.114,drugC
3,28,F,NORMAL,HIGH,7.798,drugX
4,61,F,LOW,HIGH,18.043,drugY


In [15]:
('Data shape is:', df.shape)

('Data shape is:', (200, 6))

In [16]:
df.dtypes

Age              int64
Sex             object
BP              object
Cholesterol     object
Na_to_K        float64
Drug            object
dtype: object

In [17]:
(df.isnull().sum())

Age            0
Sex            0
BP             0
Cholesterol    0
Na_to_K        0
Drug           0
dtype: int64

### PREPROCESSING AND ENCODING

In [18]:
df['Sex'] = df['Sex'].map({'M': 0, 'F': 1})
df['BP'] = df['BP'].map({'LOW': 0, 'NORMAL': 1, 'HIGH': 2})
df['Cholesterol'] = df['Cholesterol'].map({'NORMAL': 0, 'HIGH': 1})


In [19]:
(df.head())

,Age,Sex,BP,Cholesterol,Na_to_K,Drug
0,23,1,2,1,25.355,drugY
1,47,0,0,1,13.093,drugC
2,47,0,0,1,10.114,drugC
3,28,1,1,1,7.798,drugX
4,61,1,0,1,18.043,drugY


### separating X and y

In [20]:
X = df.drop('Drug', axis=1)
y = df['Drug']


### TRAIN TEST SPLIT AND SCALING


In [21]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler


In [22]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [23]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

###  TRAINING MODELS AND CHECKING ACCURACY

In [24]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report


### Model 1: Logistic Regression

In [37]:
lr = LogisticRegression(random_state=42)
lr.fit(X_train_scaled, y_train)
pred1 = lr.predict(X_test_scaled)
print("1. Logistic Regression Accuracy:", accuracy_score(y_test, pred1))


1. Logistic Regression Accuracy: 0.925


###  Model 2: Decision Tree

In [26]:
dt = DecisionTreeClassifier(random_state=42)
dt.fit(X_train_scaled, y_train)
pred2 = dt.predict(X_test_scaled)
print("2. Decision Tree Accuracy:", accuracy_score(y_test, pred2))


2. Decision Tree Accuracy: 1.0


### Model 3: Random Forest

In [38]:
rf = RandomForestClassifier(random_state=42)
rf.fit(X_train_scaled, y_train)
pred3 = rf.predict(X_test_scaled)
print("3. Random Forest Accuracy:", accuracy_score(y_test, pred3))


3. Random Forest Accuracy: 1.0


### Model 4: KNN

In [29]:
knn = KNeighborsClassifier()
knn.fit(X_train_scaled, y_train)
pred4 = knn.predict(X_test_scaled)
print("4. KNN Accuracy:", accuracy_score(y_test, pred4))


4. KNN Accuracy: 0.85


### Model 5: SVM

In [30]:
svm = SVC(random_state=42)
svm.fit(X_train_scaled, y_train)
pred5 = svm.predict(X_test_scaled)
print("5. SVM Accuracy:", accuracy_score(y_test, pred5))

5. SVM Accuracy: 0.975


### TESTING ON CUSTOM DATA 

In [31]:
test_data = pd.DataFrame([
    [23, 'F', 'HIGH', 'HIGH', 25.3],
    [47, 'M', 'LOW', 'HIGH', 13.1],
    [60, 'M', 'HIGH', 'NORMAL', 10.5],
    [35, 'F', 'NORMAL', 'NORMAL', 30.2],
    [22, 'F', 'HIGH', 'NORMAL', 8.4]
], columns=['Age', 'Sex', 'BP', 'Cholesterol', 'Na_to_K'])


### encoding the test data same way

In [32]:
test_data['Sex'] = test_data['Sex'].map({'M': 0, 'F': 1})
test_data['BP'] = test_data['BP'].map({'LOW': 0, 'NORMAL': 1, 'HIGH': 2})
test_data['Cholesterol'] = test_data['Cholesterol'].map({'NORMAL': 0, 'HIGH': 1})

### scale the test data

In [33]:
test_data_scaled = scaler.transform(test_data)

In [34]:
final_preds = dt.predict(test_data_scaled)
print("\nPredictions for 5 custom patients:")
print(final_preds)


Predictions for 5 custom patients:
['drugY' 'drugC' 'drugB' 'drugY' 'drugA']


###  USING PIPELINE

In [35]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder


X_p = df.drop('Drug', axis=1) 

ct = ColumnTransformer(transformers=[
    ('num', StandardScaler(), ['Age', 'Na_to_K']),
    ('cat', OneHotEncoder(), ['Sex', 'BP', 'Cholesterol'])
], remainder='passthrough')

pipe = Pipeline([
    ('pre', ct),
    ('model', DecisionTreeClassifier(random_state=42))
])


### re-splitting raw data just to show pipeline workflow

In [36]:
X_train_raw, X_test_raw, y_train_raw, y_test_raw = train_test_split(X, y, test_size=0.2, random_state=42)
pipe.fit(X_train_raw, y_train_raw)
pipe_pred = pipe.predict(X_test_raw)
print("\nPipeline Accuracy:", accuracy_score(y_test_raw, pipe_pred))


Pipeline Accuracy: 1.0
